# Chapter 5 — Context Engineering
## AI-Based Data Engineering (Packt)

Context engineering is the discipline of deciding *what* information to include in a prompt, in *what order*, and *how much* of it. The three levers are:

1. **Source selection** — which of the six source categories is relevant to this query?
2. **Chunking** — at what granularity do you split each source?
3. **Budget management** — how do you fit the most relevant context within the token limit?

**What you'll build:** a context assembler that pulls schema metadata, sample data, and table freshness signals from Snowflake, then passes the assembled context to `claude-haiku-4-5` for a grounded SQL generation call.

**Prerequisites:** run `code/setup/opspulse_generator.py --target snowflake` for the OPSPU.MARTS tables.

> **External Access Integration required.** This notebook calls the Anthropic API from Snowflake. Your admin must:
> 1. Create an External Access Integration for `api.anthropic.com`
> 2. Set `ANTHROPIC_API_KEY` as a Snowflake Secret and attach it to this notebook
>
> Without this, the `anthropic.Anthropic()` calls will fail. See [Snowflake EAI docs](https://docs.snowflake.com/en/developer-guide/external-network-access/creating-using-external-network-access).


In [ ]:
import anthropic
import json
from dataclasses import dataclass, field
from typing import Optional
from snowflake.snowpark.context import get_active_session

session = get_active_session()
client  = anthropic.Anthropic()
print("Session and Anthropic client ready.")

In [ ]:
%%sql -r schema_metadata
-- Source 1: Schema / catalog metadata from INFORMATION_SCHEMA
-- Authority: high  Freshness: on DDL change  Granularity: table-level
SELECT
    table_name,
    column_name,
    data_type,
    COALESCE(comment, '') AS comment
FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MARTS'
ORDER BY table_name, ordinal_position;

In [ ]:
%%sql -r sample_data
-- Source 2: Representative rows from the canonical active-customer table
-- Authority: medium  Freshness: on data change  Granularity: row-level (3-5 rows)
-- NOTE: returns empty result if FCT_ACTIVE_CUSTOMERS does not exist yet;
--       run code/setup/opspulse_generator.py --target snowflake first.
SELECT * FROM OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS LIMIT 5;

In [ ]:
%%sql -r table_freshness
-- Source 3: Table freshness signals (DDL-level approximation)
-- last_altered reflects the most recent DDL or data operation.
-- Relabeled as approx_ddl_lag to avoid overstating freshness precision.
SELECT
    table_name,
    row_count,
    last_altered
FROM OPSPU.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'MARTS'
ORDER BY last_altered DESC;

In [ ]:
# ── ContextChunk dataclass + budget-aware assembler ─────────────────────────

@dataclass
class ContextChunk:
    source:         str
    source_type:    str   # schema | sample_data | freshness | incident | policy
    content:        str
    token_estimate: int  = 0
    metadata:       dict = field(default_factory=dict)

    def __post_init__(self):
        # Rough estimate: 1 token ≈ 4 characters
        self.token_estimate = max(1, len(self.content) // 4)


def schema_rows_to_chunk(df) -> ContextChunk:
    """Convert INFORMATION_SCHEMA columns DataFrame to a ContextChunk."""
    lines = []
    for tbl, grp in df.groupby('TABLE_NAME'):
        lines.append(f'Table: OPSPU.MARTS.{tbl}')
        for _, row in grp.iterrows():
            desc = f" — {row['COMMENT']}" if row['COMMENT'] else ''
            lines.append(f"  {row['COLUMN_NAME']} ({row['DATA_TYPE']}){desc}")
        lines.append('')
    return ContextChunk(
        source='OPSPU.INFORMATION_SCHEMA.COLUMNS',
        source_type='schema',
        content='\n'.join(lines),
        metadata={'table_count': df['TABLE_NAME'].nunique()},
    )


def sample_rows_to_chunk(df, table_fqn: str) -> ContextChunk:
    """Convert sample rows DataFrame to a ContextChunk."""
    if df.empty:
        return ContextChunk(source=table_fqn, source_type='sample_data',
                            content=f'No sample rows available for {table_fqn}.')
    cols  = list(df.columns[:6])  # cap at 6 columns to control token cost
    lines = [', '.join(cols)]
    for _, row in df.head(3).iterrows():
        lines.append(', '.join(str(row[c]) for c in cols))
    return ContextChunk(
        source=table_fqn, source_type='sample_data',
        content='\n'.join(lines),
        metadata={'rows_shown': min(3, len(df))},
    )


def freshness_rows_to_chunk(df) -> ContextChunk:
    """Convert table freshness DataFrame to a ContextChunk."""
    lines = ['Table freshness (last_altered — DDL approximation):']
    for _, row in df.iterrows():
        lines.append(
            f"  {row['TABLE_NAME']}: "
            f"rows={row['ROW_COUNT']}, last_altered={row['LAST_ALTERED']}"
        )
    return ContextChunk(
        source='OPSPU.INFORMATION_SCHEMA.TABLES',
        source_type='freshness',
        content='\n'.join(lines),
    )


def assemble_context(chunks: list, token_budget: int = 4_000) -> str:
    """Budget-aware context assembly. Fills highest-priority sources first."""
    assembled   = []
    tokens_used = 0
    for chunk in chunks:
        if tokens_used + chunk.token_estimate > token_budget:
            remaining = token_budget - tokens_used
            print(f'  [budget] Skipping {chunk.source} '
                  f'({chunk.token_estimate} tokens, {remaining} remaining)')
            continue
        assembled.append(
            f'[{chunk.source_type.upper()}] {chunk.source}\n{chunk.content}'
        )
        tokens_used += chunk.token_estimate
    print(f'  [budget] Assembled {len(assembled)}/{len(chunks)} chunks, '
          f'~{tokens_used} tokens')
    return '\n\n---\n\n'.join(assembled)


print("ContextChunk and assemble_context defined.")

In [ ]:
# ── Assemble context from all three sources ─────────────────────────────────

schema_df    = schema_metadata.to_pandas()
sample_df    = sample_data.to_pandas()
freshness_df = table_freshness.to_pandas()

schema_chunk    = schema_rows_to_chunk(schema_df)
sample_chunk    = sample_rows_to_chunk(sample_df, 'OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS')
freshness_chunk = freshness_rows_to_chunk(freshness_df)

# Priority order: schema first (most authoritative), sample second, freshness third
chunks = [schema_chunk, sample_chunk, freshness_chunk]
print('Context chunks built:')
for c in chunks:
    print(f'  {c.source_type:12s} | {c.token_estimate:4d} tokens | {c.source}')

print('\nAssembling context (4,000-token budget):')
assembled = assemble_context(chunks, token_budget=4_000)
print(f'\nAssembled context preview (first 600 chars):\n{assembled[:600]}')

In [ ]:
# ── Grounded SQL generation with assembled context ──────────────────────────

response = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=400,
    system=(
        'You are a senior Snowflake data engineer. '
        'Generate read-only SELECT queries using ONLY the tables and columns '
        'described in the provided context. Never reference tables outside the context.'
    ),
    messages=[{
        'role': 'user',
        'content': (
            f'Context:\n{assembled}\n\n'
            'Question: How many active customers are in each region, '
            'and what is the average 30-day order count per customer per region?\n\n'
            'Return: one Snowflake SELECT query answering this question.'
        ),
    }],
)

print('Grounded response from claude-haiku-4-5:')
print(response.content[0].text)

In [ ]:
%%sql -r token_estimates
-- Rough token budget estimate per column metadata row
-- 1 token ≈ 4 characters — guides how many columns fit in a given budget
SELECT
    column_name,
    data_type,
    COALESCE(comment, '')                                                   AS comment,
    LENGTH(column_name || ' ' || data_type || ' ' || COALESCE(comment, '')) AS approx_chars,
    ROUND(
        LENGTH(column_name || ' ' || data_type || ' ' || COALESCE(comment, '')) / 4.0
    )                                                                       AS approx_tokens
FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MARTS'
ORDER BY approx_tokens DESC
LIMIT 10;

## Summary

| Source category | Authority | Freshness | Chunking granularity |
|---|---|---|---|
| Schema metadata | High | On DDL change | Table-level |
| Sample data | Medium | On data change | Row-level (3–5 rows) |
| Table freshness | Medium | On DDL/data | Table-level |
| dbt model artifacts | High | On dbt run | Model-level |
| Incident history | High | Real-time | Ticket-level |
| Policy documents | High | On policy change | Section-level |

`assemble_context()` respects the token budget by processing chunks in priority order — the deterministic baseline. Chapter 5 also covers retrieval-augmented approaches (Cortex Search) for large knowledge bases where priority ordering is not sufficient. See `code/ch05_context_engineering/context_sources.py` for the full implementation including dbt and Cortex Search sources.